In [2]:
!pip install -q torch scikit-learn numpy matplotlib netcal

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss
from google.colab import drive

drive.mount('/content/drive')

# Full seeding — makes every run reproducible
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

SAVE_PATH = '/content/drive/MyDrive/ptb-xl-dataset/'
CKPT_PATH = '/content/drive/MyDrive/ptb-xl-dataset/'
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load preprocessed data
X_train_norm = np.load(SAVE_PATH + 'X_train_norm.npy')
X_test_norm  = np.load(SAVE_PATH + 'X_test_norm.npy')
y_train_enc  = np.load(SAVE_PATH + 'y_train_enc.npy')
y_test_enc   = np.load(SAVE_PATH + 'y_test_enc.npy')

# Recover fold assignments (same row order as the arrays)
Y = pd.read_csv(SAVE_PATH + 'ptb-xl/ptbxl_database.csv', index_col='ecg_id')
train_meta = Y[Y.strat_fold != 10]

# Validation = fold 9, Training = folds 1-8 (patient-separated, no leakage)
val_mask   = (train_meta.strat_fold == 9).values
train_mask = (train_meta.strat_fold != 9).values

X_train, y_train = X_train_norm[train_mask], y_train_enc[train_mask]
X_val,   y_val   = X_train_norm[val_mask],   y_train_enc[val_mask]

print(f"Train:      {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test_norm.shape}")
print(f"Device:     {device}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 998.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.0/236.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.2/291.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.8/174.8 kB 8.2 MB/s eta 0:00:00
Mounted at /content/drive
Train:      (17418, 1000, 12)
Validation: (2183, 1000, 12)
Test:       (2198, 1000, 12)
Device:     cpu


In [3]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=7,
                               stride=stride, padding=3, bias=False)
        self.bn1   = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=7,
                               padding=3, bias=False)
        self.bn2   = nn.BatchNorm1d(out_channels)
        self.skip  = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.skip(x)
        return F.relu(out)


class ECGEncoder(nn.Module):
    def __init__(self, embedding_dim=256):
        super().__init__()
        self.conv1   = nn.Conv1d(12, 64, kernel_size=15, stride=2, padding=7, bias=False)
        self.bn1     = nn.BatchNorm1d(64)
        self.layer1  = ResidualBlock(64,  64)
        self.layer2  = ResidualBlock(64,  128, stride=2)
        self.layer3  = ResidualBlock(128, 256, stride=2)
        self.layer4  = ResidualBlock(256, 512, stride=2)
        self.pool    = nn.AdaptiveAvgPool1d(1)
        self.project = nn.Linear(512, embedding_dim)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).squeeze(-1)
        x = self.project(x)
        return x


class ECGDataset(Dataset):
    def __init__(self, signals, labels):
        self.signals = torch.FloatTensor(signals)
        self.labels  = torch.FloatTensor(labels)

    def __len__(self):
        return len(self.signals)

    def __getitem__(self, idx):
        return self.signals[idx], self.labels[idx]


print("Architecture defined.")

Architecture defined.


In [ ]:
# ============================================================
# Multi-seed evaluation: run both models across 5 seeds,
# report mean ± std for every metric, before and after calibration.
# ============================================================
!pip install -q netcal
from netcal.metrics import ECE

SEEDS = [1, 2, 3, 4, 5]

def set_all_seeds(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

def train_model(model, X_tr, y_tr, seed, n_epochs=20, lr=1e-4):
    dataset   = ECGDataset(X_tr, y_tr)
    g = torch.Generator(); g.manual_seed(seed)
    loader    = DataLoader(dataset, batch_size=64, shuffle=True, generator=g)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCELoss()
    model.train()
    for epoch in range(n_epochs):
        for signals, labels in loader:
            signals, labels = signals.to(device), labels.to(device)
            loss = criterion(model(signals), labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    return model

def get_logits(model, X):
    enc, lin = model[0], model[1]
    enc.eval(); lin.eval()
    loader = DataLoader(ECGDataset(X, np.zeros((len(X), 5))), batch_size=64, shuffle=False)
    out = []
    with torch.no_grad():
        for signals, _ in loader:
            out.append(lin(enc(signals.to(device))).cpu())
    return torch.cat(out)

class TemperatureScaling(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1))
    def forward(self, logits):
        return torch.sigmoid(logits / self.temperature)

def fit_temperature(model, X_val, y_val):
    logits = get_logits(model, X_val).to(device)
    labels = torch.FloatTensor(y_val).to(device)
    tm = TemperatureScaling().to(device)
    opt = optim.LBFGS([tm.temperature], lr=0.01, max_iter=50)
    crit = nn.BCELoss()
    def closure():
        opt.zero_grad()
        l = crit(tm(logits), labels); l.backward(); return l
    opt.step(closure)
    return tm

def metrics(probs, y):
    preds = (probs > 0.5).astype(int)
    auc   = roc_auc_score(y, probs, average='macro')
    f1    = f1_score(y, preds, average='macro')
    brier = np.mean([brier_score_loss(y[:, i], probs[:, i]) for i in range(5)])
    eps   = 1e-7
    nll   = -np.mean(y*np.log(probs+eps) + (1-y)*np.log(1-probs+eps))
    ece_m = ECE(10)
    ece   = float(np.mean([ece_m.measure(probs[:, c], y[:, c]) for c in range(5)]))
    return auc, f1, brier, nll, ece

def probs_from_temp(model, temp, X):
    logits = get_logits(model, X).to(device)
    with torch.no_grad():
        return temp(logits).cpu().numpy()

def probs_plain(model, X):
    model.eval()
    loader = DataLoader(ECGDataset(X, np.zeros((len(X), 5))), batch_size=64, shuffle=False)
    out = []
    with torch.no_grad():
        for signals, _ in loader:
            out.append(model(signals.to(device)).cpu().numpy())
    return np.concatenate(out)

# Collect results: results[model][stage] = list of (auc,f1,brier,nll,ece) across seeds
results = {m: {'before': [], 'after': []} for m in ['Scratch', 'CPC']}
temps   = {'Scratch': [], 'CPC': []}

for seed in SEEDS:
    print(f"===== Seed {seed} =====")
    set_all_seeds(seed)

    # Sample 10% subset (varies with seed)
    n = int(len(X_train) * 0.10)
    idx = np.random.choice(len(X_train), n, replace=False)
    X_tr, y_tr = X_train[idx], y_train[idx]

    # --- Scratch ---
    set_all_seeds(seed)
    scr = nn.Sequential(ECGEncoder(256), nn.Linear(256, 5), nn.Sigmoid()).to(device)
    scr = train_model(scr, X_tr, y_tr, seed)
    p_before = probs_plain(scr, X_test_norm)
    t = fit_temperature(scr, X_val, y_val)
    p_after = probs_from_temp(scr, t, X_test_norm)
    results['Scratch']['before'].append(metrics(p_before, y_test_enc))
    results['Scratch']['after'].append(metrics(p_after, y_test_enc))
    temps['Scratch'].append(t.temperature.item())

    # --- CPC ---
    set_all_seeds(seed)
    enc = ECGEncoder(256).to(device)
    enc.load_state_dict(torch.load(CKPT_PATH + 'cpc_checkpoint_epoch50.pt', map_location=device)['encoder_state'])
    cpc = nn.Sequential(enc, nn.Linear(256, 5), nn.Sigmoid()).to(device)
    cpc = train_model(cpc, X_tr, y_tr, seed)
    p_before = probs_plain(cpc, X_test_norm)
    t = fit_temperature(cpc, X_val, y_val)
    p_after = probs_from_temp(cpc, t, X_test_norm)
    results['CPC']['before'].append(metrics(p_before, y_test_enc))
    results['CPC']['after'].append(metrics(p_after, y_test_enc))
    temps['CPC'].append(t.temperature.item())

# ---- Print mean ± std ----
names = ['AUC', 'F1', 'Brier', 'NLL', 'ECE']
def summarise(tag):
    for model in ['Scratch', 'CPC']:
        arr = np.array(results[model][tag])
        mean, std = arr.mean(axis=0), arr.std(axis=0)
        line = " | ".join(f"{names[i]}: {mean[i]:.4f}±{std[i]:.4f}" for i in range(5))
        print(f"{model:8s} ({tag}) {line}")

print("\n================ RESULTS (mean ± std over 5 seeds) ================")
print("\n-- Before calibration --")
summarise('before')
print("\n-- After calibration --")
summarise('after')
print(f"\nTemperatures — Scratch: {np.mean(temps['Scratch']):.4f}±{np.std(temps['Scratch']):.4f} | "
      f"CPC: {np.mean(temps['CPC']):.4f}±{np.std(temps['CPC']):.4f}")

===== Seed 1 =====
===== Seed 2 =====
===== Seed 3 =====


In [ ]:
# ===== FIGURE 11 — Reliability diagram (before vs after scaling) =====
import numpy as np, torch
import matplotlib.pyplot as plt
COL_SCRATCH, COL_SIMCLR = 'steelblue', 'coral'
plt.rcParams.update({'savefig.dpi':150,'font.size':11,
                     'axes.spines.top':False,'axes.spines.right':False})

def _probs_before_after(model, temp, X):
    logits = get_logits(model, X).to(device)
    with torch.no_grad():
        return torch.sigmoid(logits).cpu().numpy(), temp(logits).cpu().numpy()

def _reliability(probs, labels, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    conf, acc = [], []
    p, y = probs.reshape(-1), labels.reshape(-1)
    for i in range(n_bins):
        hi = (p <= edges[i+1]) if i == n_bins-1 else (p < edges[i+1])
        m = (p >= edges[i]) & hi
        if m.sum() > 0:
            conf.append(p[m].mean()); acc.append(y[m].mean())
        else:
            conf.append(np.nan); acc.append(np.nan)
    return np.array(conf), np.array(acc)

p_before, p_after = _probs_before_after(cpc_model, cpc_temp, X_test_norm)
c_b, a_b = _reliability(p_before, y_test_enc)
c_a, a_a = _reliability(p_after,  y_test_enc)

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], 'k--', alpha=0.6, label='Perfect calibration')
plt.plot(c_b, a_b, 'o-', color=COL_SIMCLR,  label='Before scaling')
plt.plot(c_a, a_a, 's-', color=COL_SCRATCH, label='After scaling')
plt.xlabel("Mean predicted probability"); plt.ylabel("Observed frequency")
plt.title("Reliability Diagram — Calibrated CPC (pooled over classes)")
plt.xlim(0, 1); plt.ylim(0, 1)
plt.legend(frameon=False, loc='upper left')
plt.gca().set_aspect('equal')
plt.tight_layout()
plt.savefig("fig11_reliability_diagram.png", bbox_inches='tight')
plt.show()